# Notebook 02: Prompting Techniques

This notebook compares **zero-shot**, **few-shot**, and **chain-of-thought** prompting techniques. You will learn when to use each approach and measure their effectiveness.

## Learning Objectives
- Apply zero-shot, few-shot, and chain-of-thought prompting
- Compare output quality across techniques
- Determine which technique fits different task types
- Handle edge cases with prompting strategies

In [ ]:
import json
import openai


client = openai.OpenAI()

def generate(prompt, system_prompt=None, temperature=0.3, max_tokens=1024):
    """Generate response from Gemini."""
    config = types.GenerateContentConfig(
        temperature=temperature,
        max_output_tokens=max_tokens
    )
    if system_prompt:
        config.system_instruction = system_prompt
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}],
        config=config
    )
    return response.choices[0].message.content

print("Ready.")

## 1. Zero-Shot Prompting

No examples provided. The model relies on its pre-trained knowledge and the task description alone.

In [ ]:
# Zero-shot classification
zero_shot_prompt = """Classify the following customer message into one of these categories:
- BILLING: pricing, payments, invoices, refunds
- TECHNICAL: bugs, errors, API issues, integrations
- FEATURE: requests, suggestions, improvements
- ACCOUNT: login, password, permissions, settings

Customer message: "I keep getting a 403 error when trying to access the admin panel
even though I should have admin permissions."

Category:"""

print("=== ZERO-SHOT PROMPT ===")
print(zero_shot_prompt)
print("\n--- Response ---")
response = generate(zero_shot_prompt)
print(response)

In [ ]:
# Zero-shot with multiple examples (run separately to see consistency)
test_messages = [
    "How do I upgrade my plan?",
    "Your API is returning timeout errors on the /users endpoint",
    "It would be great if you could add dark mode support",
    "I can't reset my password, the link keeps expiring",
    "I was charged twice this month",
]

system = "Classify customer support messages into: BILLING, TECHNICAL, FEATURE, ACCOUNT. Return only the category."

print("=== ZERO-SHOT CONSISTENCY TEST ===")
for msg in test_messages:
    response = generate(f"Message: {msg}\nCategory:", system_prompt=system, temperature=0.0)
    print(f"  {msg[:50]:50s} -> {response.strip()}")

## 2. Few-Shot Prompting

Provide examples that demonstrate the expected input-output mapping.

In [ ]:
# Few-shot classification
few_shot_prompt = """Classify customer support messages into categories.

Examples:

Message: "I can't log in, it says my account is locked."
Category: ACCOUNT

Message: "The dashboard is showing wrong data for the last 3 days."
Category: TECHNICAL

Message: "Can you add integration with Salesforce?"
Category: FEATURE

Message: "Why was I charged $99 when my plan is $49?"
Category: BILLING

Now classify this message:

Message: "I keep getting a 403 error when trying to access the admin panel
even though I should have admin permissions."
Category:"""

print("=== FEW-SHOT PROMPT ===")
print(few_shot_prompt)
print("\n--- Response ---")
response = generate(few_shot_prompt, temperature=0.0)
print(response)

In [ ]:
# Few-shot with output format examples
few_shot_structured = """Extract structured data from support tickets.

Examples:

Input: "My credit card ending in 4242 was charged $199 but I'm on the Starter plan ($29/mo). Please refund the difference."
Output:
{
  "issue_type": "billing",
  "sentiment": "frustrated",
  "urgency": "high",
  "action_required": "refund",
  "customer_plan": "Starter",
  "amount_in_dispute": 170
}

Input: "The webhook integration stopped working after your last API update."
Output:
{
  "issue_type": "technical",
  "sentiment": "concerned",
  "urgency": "medium",
  "action_required": "investigate",
  "customer_plan": "unknown",
  "amount_in_dispute": 0
}

Input: "It would be nice if you could support importing from Jira."
Output:
{
  "issue_type": "feature_request",
  "sentiment": "neutral",
  "urgency": "low",
  "action_required": "log_feature_request",
  "customer_plan": "unknown",
  "amount_in_dispute": 0
}

Now extract from this input:

Input: "I was on the Enterprise plan and cancelled last week but I'm still being billed. This is the third time this year."
Output:"""

print("=== FEW-SHOT STRUCTURED EXTRACTION ===")
response = generate(few_shot_structured, temperature=0.0)
print(response)

## 3. Chain-of-Thought (CoT) Prompting

Instruct the model to reason step-by-step before producing a final answer.

In [ ]:
# Standard prompt (no CoT)
direct_prompt = """A SaaS company has:
- MRR: $500K
- Monthly churn rate: 5%
- New MRR added monthly: $80K
- COGS per customer: $15/mo
- Average revenue per customer: $50/mo

What is the net MRR growth rate and is the company sustainable?"""

print("=== DIRECT PROMPT (NO CoT) ===")
response_direct = generate(direct_prompt, temperature=0.3)
print(response_direct)

In [ ]:
# Chain-of-thought prompt
cot_prompt = """A SaaS company has:
- MRR: $500K
- Monthly churn rate: 5%
- New MRR added monthly: $80K
- COGS per customer: $15/mo
- Average revenue per customer: $50/mo

What is the net MRR growth rate and is the company sustainable?

Think through this step-by-step:
1. Calculate monthly churned MRR
2. Calculate net new MRR (new - churned)
3. Calculate growth rate as percentage of current MRR
4. Calculate gross margin per customer
5. Assess long-term sustainability based on growth rate vs churn
6. Provide final recommendation

Show your work for each step."""

print("=== CHAIN-OF-THOUGHT PROMPT ===")
response_cot = generate(cot_prompt, temperature=0.3)
print(response_cot)

In [ ]:
# CoT with structured output
cot_structured = """Analyze this code for potential issues.

Code:
```python
def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result
```

Think step-by-step:
1. Trace through the algorithm with input [3, 1, 4, 1, 5]
2. Analyze time complexity
3. Analyze space complexity
4. Identify any edge cases or bugs
5. Assess code quality and style

Final output as JSON:
{
  "correctness": "assessment",
  "time_complexity": "O(...)",
  "space_complexity": "O(...)",
  "edge_cases": ["case1"],
  "issues": ["issue1"],
  "quality_score": "1-10"
}"""

print("=== CoT + STRUCTURED OUTPUT ===")
response = generate(cot_structured, temperature=0.3)
print(response)

## 4. Technique Comparison

Run the same task with all three techniques and compare.

In [ ]:
task_input = """Review this API response handler:

def handle_response(response):
    data = response.json()
    if response.status_code == 200:
        return data['results']
    elif response.status_code == 404:
        return None
    else:
        raise Exception(f"Error: {response.status_code}")
"""

# Zero-shot
prompt_zero = f"Review this code for issues:\n{task_input}"

# Few-shot
prompt_few = f"""Review code for issues. Examples:

Code: `def divide(a, b): return a / b`
Issues: No zero-division check, no type validation

Code: `def get_user(id): return db.query(f"SELECT * FROM users WHERE id={id}")`
Issues: SQL injection vulnerability, no parameterized queries

Now review this code:
{task_input}"""

# CoT
prompt_cot = f"""Review this code step-by-step:
1. What does the function do?
2. What are the error paths?
3. What edge cases are not handled?
4. What security issues exist?
5. What improvements would you suggest?

{task_input}"""

techniques = [
    ("Zero-Shot", prompt_zero),
    ("Few-Shot", prompt_few),
    ("Chain-of-Thought", prompt_cot),
]

for name, prompt in techniques:
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    response = generate(prompt, temperature=0.3)
    print(response[:400])
    print("...")

## 5. When to Use Each Technique

| Technique | Best For | Limitations |
|-----------|----------|-------------|
| **Zero-Shot** | Simple, well-defined tasks; high-volume classification | Inconsistent format; struggles with edge cases |
| **Few-Shot** | Complex output formats; domain-specific tasks; when examples clarify ambiguity | More tokens per request; examples must be carefully selected |
| **Chain-of-Thought** | Multi-step reasoning; math/logic; complex analysis | Higher latency; more tokens; can over-explain simple tasks |
| **CoT + Few-Shot** | Complex tasks requiring specific format AND reasoning | Most expensive; use only when accuracy justifies cost |

## 6. Exercises

### Exercise 1: Zero-Shot Edge Cases

Test the zero-shot classifier with ambiguous messages that could fit multiple categories:

- "Your billing page shows an error when I try to update my credit card"
- "I need admin access to manage my team's billing"
- "The feature I requested 6 months ago still isn't available"

How consistent are the results? What happens when messages span categories?

In [ ]:
# YOUR CODE HERE
# Test zero-shot with ambiguous messages
ambiguous_messages = [
    "Your billing page shows an error when I try to update my credit card",
    "I need admin access to manage my team's billing",
    "The feature I requested 6 months ago still isn't available",
]

# Classify each message using zero-shot
# Run each message 3 times to check consistency

### Exercise 2: Design Few-Shot Examples

For a sentiment analysis task, design 4 few-shot examples that cover:
- Positive sentiment
- Negative sentiment
- Mixed sentiment
- Sarcastic sentiment (tricky!)

Test your examples with 5 new inputs.

In [ ]:
# YOUR CODE HERE
# Design few-shot examples for sentiment analysis
# Include at least one sarcastic example
# Test with 5 new inputs

### Exercise 3: Chain-of-Thought for Bug Diagnosis

Write a CoT prompt that walks through diagnosing a bug:

```
Bug report: "Users report that their session expires after 5 minutes
even though we set the session timeout to 24 hours."
```

Your CoT prompt should guide the model through:
1. Understanding the expected behavior
2. Identifying possible causes
3. Prioritizing investigation steps
4. Suggesting a fix

In [ ]:
# YOUR CODE HERE
# Write a CoT prompt for bug diagnosis
# Test it with the session expiry bug report

## Key Takeaways

1. **Zero-shot** is fast and cheap but inconsistent for complex tasks
2. **Few-shot** improves consistency by showing the model exactly what you want
3. **Chain-of-thought** improves accuracy for reasoning tasks by decomposing the problem
4. **Match technique to task complexity** — don't use CoT for simple classification
5. **Sarcastic/ambiguous inputs** often need few-shot examples to handle correctly